# 10 - Temporal Bypass Ablation for Bidirectional ConvLSTM U-Net

This notebook evaluates whether the trained bidirectional ConvLSTM U-Net depends on temporal bottleneck features or mainly uses target-frame U-Net skip connections. It reuses the same EchoNet-Dynamic test split, preprocessing, checkpoint loading, loss, threshold, and metrics used by `07_bidirectional_convlstm.ipynb`.

No model weights are modified and no retraining is performed. Ablations are applied only during inference by temporarily replacing the model's `forward` method inside a context manager, then restoring the original method.


## Kaggle / Local Setup


In [ ]:
from __future__ import annotations

from contextlib import contextmanager
from pathlib import Path
import json
import math
import os
import sys
from types import MethodType
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm


def first_existing_path(candidates):
    cleaned = [candidate for candidate in candidates if candidate]
    for candidate in cleaned:
        path = Path(candidate)
        if path.exists():
            return path
    return Path(cleaned[-1])


# Update these Kaggle input paths as needed.
PROJECT_ROOT = first_existing_path([
    os.environ.get("PROJECT_ROOT"),
    "/kaggle/input/echonet-temporal-xai",
    "/kaggle/working/Echonet_temporal_XAI",
    Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd(),
])
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.bidirectional_convlstm_unet import build_bidirectional_convlstm_unet
from src.dataset import EchoNetTemporalDataset, load_temporal_metadata, split_by_echonet_filelist
from src.temporal_train import evaluate_temporal, get_temporal_loss, segmentation_metrics
from src.utils import load_echonet_tables, set_seed

RAW_DIR = Path(os.environ.get("ECHONET_RAW_DIR", PROJECT_ROOT / "data" / "raw" / "EchoNet-Dynamic"))
PROCESSED_DIR = Path(os.environ.get("ECHONET_PROCESSED_DIR", PROJECT_ROOT / "data" / "processed"))
VIDEOS_DIR = RAW_DIR / "Videos"
CHECKPOINT_DIR = Path(os.environ.get(
    "BIDIRECTIONAL_CONVLSTM_CHECKPOINT_DIR",
    PROJECT_ROOT / "outputs" / "runs" / "bidirectional_convlstm_unet_23_frames" / "checkpoints",
))
RUN_DIR = Path("/kaggle/working/outputs/runs/temporal_bypass_ablation") if Path("/kaggle/working").exists() else PROJECT_ROOT / "outputs" / "runs" / "temporal_bypass_ablation"
FIGURES_DIR = RUN_DIR / "figures"
QUAL_DIR = FIGURES_DIR / "qualitative_examples"
MANIFEST_DIR = RUN_DIR / "manifests"

for directory in [RUN_DIR, FIGURES_DIR, QUAL_DIR, MANIFEST_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Raw EchoNet directory: {RAW_DIR}")
print(f"Processed dataset directory: {PROCESSED_DIR}")
print(f"Checkpoint directory: {CHECKPOINT_DIR}")
print(f"Output directory: {RUN_DIR}")


## Configuration

The default sequence is the trained 23-frame setup: 11 frames before, target at index 11, 11 frames after, stride 2. Keep these values synchronized with the checkpoint config.


In [ ]:
RUN_MODE = "smoke"  # change to "full" for the full official test split
SMOKE_MAX_TEST_SAMPLES = 16
FULL_MAX_TEST_SAMPLES = None

NUM_FRAMES_BEFORE = 11
NUM_FRAMES_AFTER = 11
TEMPORAL_STRIDE = 2
TARGET_IDX = NUM_FRAMES_BEFORE
SEQUENCE_LENGTH = NUM_FRAMES_BEFORE + 1 + NUM_FRAMES_AFTER
IMAGE_SIZE = (112, 112)
CHANNELS = (16, 32, 64, 128)
BATCH_SIZE = 4
NUM_WORKERS = 2
THRESHOLD = 0.5
QUALITATIVE_EXAMPLE_COUNT = 8

CONDITIONS = [
    {
        "condition": "normal",
        "description": "Normal full model inference.",
        "ablate_temporal": False,
        "ablate_skips": False,
        "ablate_forward": False,
        "ablate_backward": False,
    },
    {
        "condition": "zero_temporal_bottleneck",
        "description": "Zero fused temporal bottleneck immediately before decoder3.",
        "ablate_temporal": True,
        "ablate_skips": False,
        "ablate_forward": False,
        "ablate_backward": False,
    },
    {
        "condition": "zero_target_skips",
        "description": "Zero target-frame encoder skip1/skip2/skip3 before decoder use.",
        "ablate_temporal": False,
        "ablate_skips": True,
        "ablate_forward": False,
        "ablate_backward": False,
    },
    {
        "condition": "zero_forward_hidden",
        "description": "Zero forward ConvLSTM target-aligned hidden state before bidirectional fusion.",
        "ablate_temporal": False,
        "ablate_skips": False,
        "ablate_forward": True,
        "ablate_backward": False,
    },
    {
        "condition": "zero_backward_hidden",
        "description": "Zero backward ConvLSTM target-aligned hidden state before bidirectional fusion.",
        "ablate_temporal": False,
        "ablate_skips": False,
        "ablate_forward": False,
        "ablate_backward": True,
    },
    {
        "condition": "zero_temporal_and_skips",
        "description": "Zero fused temporal bottleneck and all target-frame encoder skips.",
        "ablate_temporal": True,
        "ablate_skips": True,
        "ablate_forward": False,
        "ablate_backward": False,
    },
]

config = {
    "run_mode": RUN_MODE,
    "seed": 42,
    "num_frames_before": NUM_FRAMES_BEFORE,
    "num_frames_after": NUM_FRAMES_AFTER,
    "temporal_stride": TEMPORAL_STRIDE,
    "target_idx": TARGET_IDX,
    "sequence_length": SEQUENCE_LENGTH,
    "image_size": list(IMAGE_SIZE),
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "threshold": THRESHOLD,
    "channels": list(CHANNELS),
    "conditions": CONDITIONS,
}
with (RUN_DIR / "config.json").open("w", encoding="utf-8") as file:
    json.dump(config, file, indent=2)
config


## Load Official Test Split

This mirrors notebook 07: load processed temporal metadata, reconstruct the official EchoNet TRAIN/VAL/TEST split using `FileList.csv`, and evaluate only the held-out TEST split.


In [ ]:
metadata_path = PROCESSED_DIR / "metadata.csv"
file_list_path = RAW_DIR / "FileList.csv"
assert metadata_path.exists(), f"Processed metadata not found: {metadata_path}"
assert file_list_path.exists(), f"EchoNet FileList.csv not found: {file_list_path}"
assert VIDEOS_DIR.exists(), f"Videos directory not found: {VIDEOS_DIR}"

samples = load_temporal_metadata(metadata_path)
file_list, _ = load_echonet_tables(RAW_DIR)
train_samples, val_samples, test_samples = split_by_echonet_filelist(samples, file_list)
matched_count = len(train_samples) + len(val_samples) + len(test_samples)
assert matched_count == len(samples), f"{len(samples) - matched_count} samples did not match the official EchoNet split."

full_test_count = len(test_samples)
if RUN_MODE == "smoke":
    test_samples = test_samples[:SMOKE_MAX_TEST_SAMPLES]
elif FULL_MAX_TEST_SAMPLES is not None:
    test_samples = test_samples[:FULL_MAX_TEST_SAMPLES]

print(f"Full official test samples: {full_test_count:,}")
print(f"Active test samples: {len(test_samples):,}")


## DataLoader

The dataset and dataloader use the same `EchoNetTemporalDataset` preprocessing as notebook 07.


In [ ]:
dataset_kwargs = {
    "videos_dir": VIDEOS_DIR,
    "num_frames_before": config["num_frames_before"],
    "num_frames_after": config["num_frames_after"],
    "temporal_stride": config["temporal_stride"],
    "image_size": tuple(config["image_size"]),
}
test_dataset = EchoNetTemporalDataset(test_samples, augment=False, **dataset_kwargs)
loader_kwargs = {
    "batch_size": config["batch_size"],
    "num_workers": config["num_workers"],
    "pin_memory": torch.cuda.is_available(),
    "persistent_workers": config["num_workers"] > 0,
}
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)

sample = test_dataset[0]
assert sample["sequence"].shape == (config["sequence_length"], 1, *config["image_size"])
assert sample["mask"].shape == (1, *config["image_size"])
assert int(sample["target_idx"]) == config["target_idx"]
assert int(sample["frame_indices"][config["target_idx"]]) == int(sample["frame_idx"])
print(f"Sample sequence shape: {tuple(sample['sequence'].shape)}")
print(f"Target index: {sample['target_idx']}")
print(f"Frame indices: {sample['frame_indices'].tolist()}")


## Load Best Checkpoint


In [ ]:
def checkpoint_config(checkpoint_path: Path) -> dict[str, Any]:
    for path in [checkpoint_path.parent.parent / "config.json", checkpoint_path.parent / "config.json", CHECKPOINT_DIR.parent / "config.json"]:
        if path.exists():
            with path.open("r", encoding="utf-8") as file:
                return json.load(file)
    return {}


def select_best_checkpoint(checkpoint_dir: Path) -> Path:
    exact = checkpoint_dir / "best_model.pt"
    if exact.exists():
        return exact
    candidates = sorted(path for path in checkpoint_dir.iterdir() if path.suffix in {".pt", ".pth", ".ckpt"})
    best_like = [path for path in candidates if "best" in path.name.lower()]
    if len(best_like) == 1:
        return best_like[0]
    if candidates:
        return candidates[0]
    raise FileNotFoundError(f"No checkpoint files found in {checkpoint_dir}")


CHECKPOINT_PATH = select_best_checkpoint(CHECKPOINT_DIR)
ckpt_cfg = checkpoint_config(CHECKPOINT_PATH)
if ckpt_cfg:
    print("Checkpoint config:")
    print(json.dumps(ckpt_cfg, indent=2))
    assert int(ckpt_cfg.get("num_frames_before", NUM_FRAMES_BEFORE)) == NUM_FRAMES_BEFORE
    assert int(ckpt_cfg.get("num_frames_after", NUM_FRAMES_AFTER)) == NUM_FRAMES_AFTER
    assert int(ckpt_cfg.get("temporal_stride", TEMPORAL_STRIDE)) == TEMPORAL_STRIDE
    assert int(ckpt_cfg.get("sequence_length", SEQUENCE_LENGTH)) == SEQUENCE_LENGTH
    channels = tuple(ckpt_cfg.get("channels", CHANNELS))
else:
    channels = CHANNELS

model = build_bidirectional_convlstm_unet(
    in_channels=1,
    out_channels=1,
    channels=tuple(channels),
    num_frames_before=NUM_FRAMES_BEFORE,
    num_frames_after=NUM_FRAMES_AFTER,
).to(device)
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
state_dict = checkpoint.get("model_state_dict", checkpoint.get("state_dict", checkpoint))
if any(key.startswith("module.") for key in state_dict):
    state_dict = {key.removeprefix("module."): value for key, value in state_dict.items()}
model.load_state_dict(state_dict)
model.eval()
loss_fn = get_temporal_loss()

print(f"Loaded checkpoint: {CHECKPOINT_PATH}")


## Inference-Time Ablation Context

The context manager below temporarily replaces `model.forward`. It reuses the trained module's own encoder, ConvLSTM cells, bidirectional fusion, decoder blocks, and output head. Ablation flags only zero selected tensors during the forward pass; weights are never edited.


In [ ]:
def tensor_max_abs(tensor: torch.Tensor) -> float:
    return float(tensor.detach().abs().amax().cpu().item())


@contextmanager
def temporal_bypass_ablation(
    model: torch.nn.Module,
    *,
    ablate_temporal: bool = False,
    ablate_skips: bool = False,
    ablate_forward: bool = False,
    ablate_backward: bool = False,
):
    original_forward = model.forward
    assertions: dict[str, Any] = {
        "ablate_temporal": ablate_temporal,
        "ablate_skips": ablate_skips,
        "ablate_forward": ablate_forward,
        "ablate_backward": ablate_backward,
        "calls": 0,
        "checks": [],
    }

    def ablated_forward(self, sequence: torch.Tensor) -> torch.Tensor:
        self._validate_sequence(sequence)
        bottlenecks, target_skips = self._encode_sequence(sequence)
        forward_hidden = self._run_temporal_branch(
            self.forward_temporal_bottleneck,
            bottlenecks,
            range(0, self.target_idx + 1),
        )
        backward_hidden = self._run_temporal_branch(
            self.backward_temporal_bottleneck,
            bottlenecks,
            range(self.expected_sequence_length - 1, self.target_idx - 1, -1),
        )

        checks: dict[str, float | bool] = {}
        if ablate_forward:
            forward_hidden = torch.zeros_like(forward_hidden)
            checks["forward_hidden_zero_max_abs"] = tensor_max_abs(forward_hidden)
            assert torch.count_nonzero(forward_hidden).item() == 0
        if ablate_backward:
            backward_hidden = torch.zeros_like(backward_hidden)
            checks["backward_hidden_zero_max_abs"] = tensor_max_abs(backward_hidden)
            assert torch.count_nonzero(backward_hidden).item() == 0

        hidden = self.bidirectional_fusion(torch.cat([forward_hidden, backward_hidden], dim=1))
        if ablate_temporal:
            hidden = torch.zeros_like(hidden)
            checks["temporal_bottleneck_zero_max_abs"] = tensor_max_abs(hidden)
            assert torch.count_nonzero(hidden).item() == 0

        skip1, skip2, skip3 = target_skips
        if ablate_skips:
            skip1 = torch.zeros_like(skip1)
            skip2 = torch.zeros_like(skip2)
            skip3 = torch.zeros_like(skip3)
            checks["skip1_zero_max_abs"] = tensor_max_abs(skip1)
            checks["skip2_zero_max_abs"] = tensor_max_abs(skip2)
            checks["skip3_zero_max_abs"] = tensor_max_abs(skip3)
            assert torch.count_nonzero(skip1).item() == 0
            assert torch.count_nonzero(skip2).item() == 0
            assert torch.count_nonzero(skip3).item() == 0

        x = self.decoder3(hidden, skip3)
        x = self.decoder2(x, skip2)
        x = self.decoder1(x, skip1)
        assertions["calls"] += 1
        assertions["checks"].append(checks)
        return self.output(x)

    try:
        model.forward = MethodType(ablated_forward, model)
        yield assertions
    finally:
        model.forward = original_forward


def condition_context(model: torch.nn.Module, condition: dict[str, Any]):
    return temporal_bypass_ablation(
        model,
        ablate_temporal=bool(condition.get("ablate_temporal", False)),
        ablate_skips=bool(condition.get("ablate_skips", False)),
        ablate_forward=bool(condition.get("ablate_forward", False)),
        ablate_backward=bool(condition.get("ablate_backward", False)),
    )


def assert_ablation_context_used(condition: dict[str, Any], assertions: dict[str, Any]) -> None:
    assert assertions["calls"] > 0, f"Forward was not called for condition {condition['condition']}"
    if condition.get("ablate_temporal"):
        assert any("temporal_bottleneck_zero_max_abs" in check for check in assertions["checks"])
        assert max(check.get("temporal_bottleneck_zero_max_abs", 0.0) for check in assertions["checks"]) == 0.0
    if condition.get("ablate_skips"):
        for key in ["skip1_zero_max_abs", "skip2_zero_max_abs", "skip3_zero_max_abs"]:
            assert any(key in check for check in assertions["checks"])
            assert max(check.get(key, 0.0) for check in assertions["checks"]) == 0.0
    if condition.get("ablate_forward"):
        assert any("forward_hidden_zero_max_abs" in check for check in assertions["checks"])
        assert max(check.get("forward_hidden_zero_max_abs", 0.0) for check in assertions["checks"]) == 0.0
    if condition.get("ablate_backward"):
        assert any("backward_hidden_zero_max_abs" in check for check in assertions["checks"])
        assert max(check.get("backward_hidden_zero_max_abs", 0.0) for check in assertions["checks"]) == 0.0



## Summary Evaluation Using Notebook 07 Pipeline

For every inference condition, this cell calls `evaluate_temporal(model, test_loader, loss_fn, device)`, the same evaluation helper used in notebook 07. The only difference is the temporary inference-time ablation context.


In [ ]:
summary_rows = []
ablation_assertion_rows = []
for condition in CONDITIONS:
    with condition_context(model, condition) as assertions:
        metrics = evaluate_temporal(model, test_loader, loss_fn, device)
    assert_ablation_context_used(condition, assertions)
    row = {
        "condition": condition["condition"],
        "description": condition["description"],
        "loss": metrics["loss"],
        "dice": metrics["dice"],
        "iou": metrics["iou"],
        "num_forward_calls": assertions["calls"],
    }
    summary_rows.append(row)
    ablation_assertion_rows.append({
        "condition": condition["condition"],
        "num_forward_calls": assertions["calls"],
        "example_checks": json.dumps(assertions["checks"][:2]),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(MANIFEST_DIR / "summary_metrics.csv", index=False)
pd.DataFrame(ablation_assertion_rows).to_csv(MANIFEST_DIR / "ablation_assertions.csv", index=False)
display(summary_df)


## Per-Sample Metrics

This cell computes per-sample Dice/IoU/loss under each condition using the same `segmentation_metrics` and loss function as the training/evaluation pipeline. Outputs are saved to CSV for downstream analysis.


In [ ]:
@torch.no_grad()
def per_sample_metrics_for_condition(condition: dict[str, Any]) -> pd.DataFrame:
    rows = []
    model.eval()
    with condition_context(model, condition) as assertions:
        for batch in tqdm(test_loader, desc=f"per-sample {condition['condition']}", leave=False):
            sequences = batch["sequence"].to(device, non_blocking=True)
            masks = batch["mask"].to(device, non_blocking=True)
            logits = model(sequences)
            loss_per_batch = loss_fn(logits, masks)
            dice, iou = segmentation_metrics(logits, masks, threshold=THRESHOLD)
            pred = (torch.sigmoid(logits) >= THRESHOLD).float()
            ids = batch.get("id", [f"sample_{i}" for i in range(sequences.shape[0])])
            video_ids = batch.get("video_id", [""] * sequences.shape[0])
            frame_indices = batch["frame_indices"].detach().cpu().numpy()
            target_frame_idx = batch["frame_idx"].detach().cpu().numpy()
            pred_area = pred.sum(dim=(1, 2, 3)).detach().cpu().numpy()
            gt_area = (masks > 0.5).float().sum(dim=(1, 2, 3)).detach().cpu().numpy()
            for i in range(sequences.shape[0]):
                rows.append({
                    "condition": condition["condition"],
                    "sample_id": str(ids[i]),
                    "video_id": str(video_ids[i]),
                    "target_frame_idx": int(target_frame_idx[i]),
                    "target_position": TARGET_IDX,
                    "sampled_frame_indices": " ".join(str(int(x)) for x in frame_indices[i]),
                    "dice": float(dice[i].detach().cpu().item()),
                    "iou": float(iou[i].detach().cpu().item()),
                    "batch_loss": float(loss_per_batch.detach().cpu().item()),
                    "predicted_foreground_area": int(pred_area[i]),
                    "ground_truth_foreground_area": int(gt_area[i]),
                })
    assert_ablation_context_used(condition, assertions)
    return pd.DataFrame(rows)


per_sample_df = pd.concat([per_sample_metrics_for_condition(condition) for condition in CONDITIONS], ignore_index=True)
per_sample_df.to_csv(MANIFEST_DIR / "per_sample_metrics.csv", index=False)

comparison_df = per_sample_df.pivot_table(
    index=["sample_id", "video_id", "target_frame_idx"],
    columns="condition",
    values=["dice", "iou"],
    aggfunc="last",
)
comparison_df.to_csv(MANIFEST_DIR / "per_sample_condition_comparison.csv")
display(per_sample_df.head())
display(comparison_df.head())


## Condition-Level Comparison Table


In [ ]:
normal = summary_df.loc[summary_df["condition"] == "normal"].iloc[0]
comparison_rows = []
for row in summary_df.to_dict("records"):
    comparison_rows.append({
        **row,
        "delta_dice_vs_normal": row["dice"] - normal["dice"],
        "delta_iou_vs_normal": row["iou"] - normal["iou"],
        "delta_loss_vs_normal": row["loss"] - normal["loss"],
    })
condition_comparison_df = pd.DataFrame(comparison_rows)
condition_comparison_df.to_csv(MANIFEST_DIR / "condition_comparison.csv", index=False)
display(condition_comparison_df)


## Qualitative Examples


In [ ]:
@torch.no_grad()
def collect_qualitative_examples(max_examples: int = 8) -> dict[str, dict[str, Any]]:
    examples: dict[str, dict[str, Any]] = {}
    for batch in test_loader:
        sequences = batch["sequence"].to(device, non_blocking=True)
        masks = batch["mask"].to(device, non_blocking=True)
        ids = batch.get("id", [f"sample_{i}" for i in range(sequences.shape[0])])
        for condition in CONDITIONS:
            with condition_context(model, condition) as assertions:
                logits = model(sequences)
            assert_ablation_context_used(condition, assertions)
            probs = torch.sigmoid(logits).detach().cpu().numpy()
            preds = (probs >= THRESHOLD).astype(np.float32)
            for i in range(sequences.shape[0]):
                sample_id = str(ids[i])
                if sample_id not in examples:
                    examples[sample_id] = {
                        "center_frame": sequences[i, TARGET_IDX, 0].detach().cpu().numpy(),
                        "ground_truth": masks[i, 0].detach().cpu().numpy(),
                        "predictions": {},
                    }
                examples[sample_id]["predictions"][condition["condition"]] = preds[i, 0]
        if len(examples) >= max_examples:
            return dict(list(examples.items())[:max_examples])
    return examples


def save_qualitative_figure(sample_id: str, example: dict[str, Any], output_path: Path) -> None:
    condition_names = [condition["condition"] for condition in CONDITIONS]
    panels = [("target frame", example["center_frame"], "gray"), ("ground truth", example["ground_truth"], "gray")]
    panels.extend((name, example["predictions"][name], "gray") for name in condition_names)
    cols = min(4, len(panels))
    rows = int(math.ceil(len(panels) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.2, rows * 3.2), squeeze=False)
    for idx, axis in enumerate(axes.flat):
        axis.axis("off")
        if idx < len(panels):
            title, image, cmap = panels[idx]
            axis.imshow(image, cmap=cmap, vmin=0, vmax=1)
            axis.set_title(title, fontsize=9)
    fig.suptitle(sample_id, fontsize=11)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close(fig)


examples = collect_qualitative_examples(QUALITATIVE_EXAMPLE_COUNT)
qual_rows = []
for sample_id, example in examples.items():
    path = QUAL_DIR / f"{sample_id}_temporal_bypass_ablation.png"
    save_qualitative_figure(sample_id, example, path)
    qual_rows.append({"sample_id": sample_id, "figure_path": str(path.relative_to(RUN_DIR))})
qualitative_df = pd.DataFrame(qual_rows)
qualitative_df.to_csv(MANIFEST_DIR / "qualitative_examples.csv", index=False)
display(qualitative_df)


## Expected Interpretation

If zeroing the fused temporal bottleneck barely changes Dice while zeroing the target-frame skips sharply degrades Dice, the model is primarily relying on target-frame spatial encoder skip features. If zeroing temporal bottleneck features strongly degrades Dice, temporal context contributes materially. The forward-only and backward-only ablations help identify directional asymmetry in temporal contribution.
